# KKBox Subscription Survival Analysis - Data Preparation

## Business Problem

**Goal:** Understand customer churn patterns in a subscription business to:
- Identify when customers are most likely to churn
- Measure how long subscriptions typically last
- Determine which factors (pricing, auto-renewal, demographics) affect churn risk
- Build predictive models to identify at-risk customers

## What is Survival Analysis?

**Survival analysis** models "time-to-event" - in our case, time until a customer churns.

**Key concepts:**
- **Event:** Customer churn (stops subscribing)
- **Censoring:** Customers still active when observation ends (we don't know IF/WHEN they'll churn)
- **Survival time:** Duration from subscription start until churn (or censoring)

**Why it's better than simple churn rate:**
- Accounts for time dimension (not just "churned yes/no")
- Handles censored observations properly (customers who haven't churned yet)
- Identifies WHEN churn happens (not just IF)
- Can incorporate time-varying factors

## The Dataset: KKBox Music Streaming

**Source:** Kaggle KKBox Churn Prediction Challenge (2018)

**Data files:**
- `transactions_v2.csv` - Subscription transaction logs (through March 31, 2017)

**Key fields:**
- `msno` - Customer ID
- `transaction_date` - When subscription transaction occurred
- `membership_expire_date` - When membership expires
- `payment_plan_days` - Length of subscription plan purchased
- `is_auto_renew` - Whether auto-renewal is enabled
- `is_cancel` - Whether customer cancelled in this transaction


## 4-Step Data Preparation

- Step 1: Load Raw Data

- Step 2: Clean Membership Events

- Step 3: Identify Churn Events

- Step 4: Derive Subscriptions

## Expected Output

A clean subscriptions dataset where:
- Each row = one complete subscription period
- Includes start date, end date, duration, and churn flag
- Users who churn and return have multiple rows (one per subscription)
- Ready for Kaplan-Meier curves and Cox regression

**Format:**
```
subscription_id | msno | starts_at | ends_at | duration_days | churned | prior_subscriptions
1               | A123 | 2015-01   | 2015-04 | 91           | 1       | 0
2               | A123 | 2015-12   | 2016-01 | 36           | 1       | 1
3               | B456 | 2016-03   | NULL    | 397          | 0       | 0
```

## Key Definitions

- **Churn:** Customer lets membership expire and doesn't renew within 30 days

- **Censored:** Customer still has active membership at observation end (2017-04-01)

- **Subscription:** Continuous period from first transaction until churn (or censoring)

- **Reactivation:** User who churned previously comes back (creates new subscription)

---

## Let's Begin!

In [58]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime

## STEP 1 - Load Data

In [59]:
# Load transactions data
transactions = pd.read_csv('../data/transactions_v2.csv')

In [60]:
# Display the first few rows of the transactions data
transactions.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,32,90,298,298,0,20170131,20170504,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,41,30,149,149,1,20150809,20190412,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,36,30,180,180,1,20170303,20170422,0
3,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
4,+00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=,41,30,99,99,1,20170323,20170423,0


In [61]:
# Check the number of records
print("Number of records in transactions data:", len(transactions))

Number of records in transactions data: 1431009


In [62]:
# Date should end at 2017-03-31 (end of Q1 2017)
print(f"  Date range: {transactions['transaction_date'].min()} to {transactions['transaction_date'].max()}")

# Parse dates
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'], 
    format='%Y%m%d'
)
transactions['membership_expire_date'] = pd.to_datetime(
    transactions['membership_expire_date'], 
    format='%Y%m%d'
)

  Date range: 20150101 to 20170331


In [63]:
# Check data types and missing values
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1431009 entries, 0 to 1431008
Data columns (total 9 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   msno                    1431009 non-null  object        
 1   payment_method_id       1431009 non-null  int64         
 2   payment_plan_days       1431009 non-null  int64         
 3   plan_list_price         1431009 non-null  int64         
 4   actual_amount_paid      1431009 non-null  int64         
 5   is_auto_renew           1431009 non-null  int64         
 6   transaction_date        1431009 non-null  datetime64[ns]
 7   membership_expire_date  1431009 non-null  datetime64[ns]
 8   is_cancel               1431009 non-null  int64         
dtypes: datetime64[ns](2), int64(6), object(1)
memory usage: 98.3+ MB


## STEP 2 - Clean Membership Events

The transaction log is messy. We need to clean it by:
1. Filtering invalid transactions
2. Consolidating same-day duplicates
3. Fixing backdated expiries
4. Adding dummy transactions

### Invalid payment plans

In [64]:
# Check if any payment_plan_days values are less than 0
negative_payment_days = (transactions['payment_plan_days'] < 0).sum()
print(f"Transactions with payment_plan_days < 0: {negative_payment_days}")

# Also check for zero values
zero_payment_days = (transactions['payment_plan_days'] == 0).sum()
print(f"Transactions with payment_plan_days == 0: {zero_payment_days}")

# Show distribution summary
print(f"\nPayment plan days statistics:")
print(transactions['payment_plan_days'].describe())

Transactions with payment_plan_days < 0: 0
Transactions with payment_plan_days == 0: 2218

Payment plan days statistics:
count    1.431009e+06
mean     6.601770e+01
std      1.024864e+02
min      0.000000e+00
25%      3.000000e+01
50%      3.000000e+01
75%      3.000000e+01
max      4.500000e+02
Name: payment_plan_days, dtype: float64


Keep only transactions where `payment_plan_days > 0`.

In [65]:
# Remove invalid transactions
transactions_valid = transactions[transactions['payment_plan_days'] > 0].copy()
print(f"   Kept {len(transactions_valid):,} of {len(transactions):,} transactions")

   Kept 1,428,791 of 1,431,009 transactions


### Same-day transactions

In [70]:
# Check for multiple transactions on the same date per user
duplicate_dates = transactions_valid.groupby(['msno', 'transaction_date']).size()
multiple_transactions = (duplicate_dates > 1).sum()

print(f"User-date combinations with multiple transactions: {multiple_transactions:,}")
print(f"Total user-date combinations: {len(duplicate_dates):,}")

# Show distribution of transactions per date
print("\nDistribution of transactions per user-date:")
print(duplicate_dates.value_counts().sort_index().head(10))

# Show an example of multiple transactions on same date
if multiple_transactions > 0:
  sample_duplicates = transactions_valid[
    transactions_valid.duplicated(subset=['msno', 'transaction_date'], keep=False)
  ].sort_values(['msno', 'transaction_date']).head(10)
  
  print("\nExample of multiple transactions on same date:")
  print(sample_duplicates[['msno', 'transaction_date', 'membership_expire_date']])

User-date combinations with multiple transactions: 27,514
Total user-date combinations: 1,395,960

Distribution of transactions per user-date:
1     1368446
2       25802
3        1199
4         190
5          56
6          72
7          28
8          22
9          27
10         13
Name: count, dtype: int64

Example of multiple transactions on same date:
                                                 msno transaction_date  \
1037325  ++Z7/94w4+ZvUIgyOWeMzefNr1EYmXfOg86g23FKgPo=       2017-03-27   
1137205  ++Z7/94w4+ZvUIgyOWeMzefNr1EYmXfOg86g23FKgPo=       2017-03-27   
150171   ++kosgi4V03jOxcBKjM/9tPignUOxcc7jBVnZLJ+lX0=       2017-03-24   
1316714  ++kosgi4V03jOxcBKjM/9tPignUOxcc7jBVnZLJ+lX0=       2017-03-24   
171355   ++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=       2015-01-09   
443083   ++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=       2015-01-09   
249717   ++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=       2015-02-09   
1244979  ++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOe


Multiple transactions can occur on the same date with different expiry dates. We don't have timestamps, so we can't tell which happened last. For each transaction date, we take the MAX (furthest) expiry date as the final subscription state.

In [51]:
# Group by date and take MAX expiry date for each user-date
transactions_consolidated = transactions_valid.groupby(['msno', 'transaction_date']).agg({
    'membership_expire_date': 'max'
}).reset_index()

transactions_consolidated.columns = ['msno', 'trans_at', 'expires_at']
print(f"   Consolidated to {len(transactions_consolidated):,} unique transaction-date records")

   Consolidated to 1,395,960 unique transaction-date records


### Backdated expiry dates

In [52]:
# Check if expiry is before transaction
backdated_count = (transactions_consolidated['expires_at'] < transactions_consolidated['trans_at']).sum()
print(f"   Found {backdated_count:,} backdated records")

   Found 4,615 backdated records


Sometimes `membership_expire_date < transaction_date` (expires in the past). This is illogical!
In such instances, we reset `expires_at = trans_at`. This ensures subscription is at least valid on the transaction date and is a conservative approach compared to dropping affeted records.

In [54]:
transactions_consolidated['expires_at'] = transactions_consolidated.apply(
    lambda row: row['trans_at'] if row['expires_at'] < row['trans_at'] else row['expires_at'],
    axis=1)

### Defining Churn - 30-day grace period

> A customer is only considered churned if their membership expires AND they don't renew within 30 days of expiration.

Since our observation period ends March 31, 2017, we add a dummy transaction on April 1, 2017 for all users. This prevents us from incorrectly marking users as churned if their membership expired in late March but the 30-day window hasn't passed yet.

In [57]:
# Add dummy April 1, 2017 transactions for all users
unique_users = transactions_consolidated['msno'].unique()
dummy_transactions = pd.DataFrame({
    'msno': unique_users,
    'trans_at': pd.Timestamp('2017-04-01'),
    'expires_at': pd.Timestamp('2017-04-01')
})

# Combine real and dummy transactions
membership_events = pd.concat([transactions_consolidated, dummy_transactions], ignore_index=True)
membership_events = membership_events.sort_values(['msno', 'trans_at', 'expires_at'])

print(f"Final events dataset: {len(membership_events):,} records")
print(f"Covering {membership_events['msno'].nunique():,} unique users")

# Display summary
print("\nSample of final events:")
print(membership_events.head(20))

Final events dataset: 2,592,992 records
Covering 1,197,032 unique users

Sample of final events:
                                                 msno   trans_at expires_at
0        +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2016-10-23 2018-02-06
1395960  +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2017-04-01 2017-04-01
1        +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-03-15 2017-04-15
1395961  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-04-01 2017-04-01
2        +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-02-28 2017-04-19
3        +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-03-31 2017-05-19
1395962  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-04-01 2017-04-01
4        +++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= 2017-03-26 2017-04-26
1395963  +++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= 2017-04-01 2017-04-01
5        ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= 2017-03-15 2017-04-15
1395964  ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= 2017-04-01 20

## STEP 3 - Identify Churn Events

Now we need to identify when users churned by:
1. Comparing consecutive transactions
2. Filtering to meaningful events
3. Calculating gaps between transactions
4. Flagging churn
5. Adjusting flag position

### The Logic

```
Transaction timeline:
├─ Trans 1: expires 2015-02-15
├─ Trans 2: expires 2015-03-15  (gap = -5 days, renewed early)
├─ Trans 3: expires 2015-04-15  (gap = -3 days, renewed early)
├─ ... 253 days pass ...
└─ Trans 4: expires 2015-12-24  (gap = 253 days, CHURNED!)
```

### Compare previous transactions

For each transaction, we need to get the expiry date from the previous transaction to know if the user renewed their subscription before it expired, or if they let it lapse.

In [73]:
# Sort all membership events
membership_events_sorted = membership_events.sort_values(['msno', 'trans_at']).copy()

# Get previous expiry date for each transaction
membership_events_sorted['previous_expires_at'] = membership_events_sorted.groupby('msno')['expires_at'].shift(1)

# First transaction has no previous and will be NaN
print("\nWith previous expiry dates:")
print(membership_events_sorted[['trans_at', 'previous_expires_at', 'expires_at']].head(20))


With previous expiry dates:
          trans_at previous_expires_at expires_at
0       2016-10-23                 NaT 2018-02-06
1395960 2017-04-01          2018-02-06 2017-04-01
1       2017-03-15                 NaT 2017-04-15
1395961 2017-04-01          2017-04-15 2017-04-01
2       2017-02-28                 NaT 2017-04-19
3       2017-03-31          2017-04-19 2017-05-19
1395962 2017-04-01          2017-05-19 2017-04-01
4       2017-03-26                 NaT 2017-04-26
1395963 2017-04-01          2017-04-26 2017-04-01
5       2017-03-15                 NaT 2017-04-15
1395964 2017-04-01          2017-04-15 2017-04-01
6       2017-02-28                 NaT 2017-04-23
7       2017-03-31          2017-04-23 2017-05-23
1395965 2017-04-01          2017-05-23 2017-04-01
8       2017-02-28                 NaT 2017-04-04
9       2017-03-31          2017-04-04 2017-05-04
1395966 2017-04-01          2017-05-04 2017-04-01
10      2016-07-04                 NaT 2017-08-21
1395967 2017-04-01   

### Identify meaningful events

- First transaction (no previous) → meaningful 
- Expiry date changed from previous → meaningful 
- Expiry date same as previous → NOT meaningful

In [84]:
# Flag meaningful events
membership_events_sorted['meaningful_event'] = 0  # Default to not meaningful

# First transaction per user
membership_events_sorted.loc[
    membership_events_sorted['previous_expires_at'].isna(), 
    'meaningful_event'
] = 1

# Expiry changed
membership_events_sorted.loc[
    membership_events_sorted['expires_at'] != membership_events_sorted['previous_expires_at'], 
    'meaningful_event'
] = 1

# Filter to meaningful only
meaningful_only = membership_events_sorted[
    membership_events_sorted['meaningful_event'] == 1
].copy()

print(f"Meaningful events: {len(meaningful_only):,} of {len(membership_events_sorted):,}")
print(membership_events_sorted[['trans_at', 'previous_expires_at', 'expires_at', 'meaningful_event']].head(20))

Meaningful events: 2,561,755 of 2,592,992
          trans_at previous_expires_at expires_at  meaningful_event
0       2016-10-23                 NaT 2018-02-06                 1
1395960 2017-04-01          2018-02-06 2017-04-01                 1
1       2017-03-15                 NaT 2017-04-15                 1
1395961 2017-04-01          2017-04-15 2017-04-01                 1
2       2017-02-28                 NaT 2017-04-19                 1
3       2017-03-31          2017-04-19 2017-05-19                 1
1395962 2017-04-01          2017-05-19 2017-04-01                 1
4       2017-03-26                 NaT 2017-04-26                 1
1395963 2017-04-01          2017-04-26 2017-04-01                 1
5       2017-03-15                 NaT 2017-04-15                 1
1395964 2017-04-01          2017-04-15 2017-04-01                 1
6       2017-02-28                 NaT 2017-04-23                 1
7       2017-03-31          2017-04-23 2017-05-23                 1
139596

### Calculate days since previous expiry

For each transaction, calculate how many days passed since the previous expiry.

- Negative days = User renewed BEFORE expiry (good)
- 0-30 days = User renewed shortly after expiry (acceptable)
- Greater than 30 days = User let it lapse → **CHURNED!**

In [77]:
# Calculate days since previous expiry
meaningful_only['days_since_expiry'] = (
    meaningful_only['trans_at'] - meaningful_only['previous_expires_at']
).dt.days

print("\nDays between previous expiry and current transaction:")
print(meaningful_only[[
    'trans_at', 'previous_expires_at', 'expires_at', 'days_since_expiry'
]].head(10))


Days between previous expiry and current transaction:
          trans_at previous_expires_at expires_at  days_since_expiry
0       2016-10-23                 NaT 2018-02-06                NaN
1395960 2017-04-01          2018-02-06 2017-04-01             -311.0
1       2017-03-15                 NaT 2017-04-15                NaN
1395961 2017-04-01          2017-04-15 2017-04-01              -14.0
2       2017-02-28                 NaT 2017-04-19                NaN
3       2017-03-31          2017-04-19 2017-05-19              -19.0
1395962 2017-04-01          2017-05-19 2017-04-01              -48.0
4       2017-03-26                 NaT 2017-04-26                NaN
1395963 2017-04-01          2017-04-26 2017-04-01              -25.0
5       2017-03-15                 NaT 2017-04-15                NaN


### Flag Churn Events

**Churn Rule:** If `days_since_previous_expiry > 30`, the user churned.

**Note:** The churn flag appears on the transaction after the gap (the reactivation). We'll fix this in the next step.

In [80]:
# Flag churn (gap > 30 days)
meaningful_only['churn_event'] = (meaningful_only['days_since_expiry'] > 30).astype(int)

print("\nWith churn event flags:")
print(meaningful_only[[
    'trans_at', 'previous_expires_at', 'expires_at', 
    'days_since_expiry', 'churn_event']].head(10))


With churn event flags:
          trans_at previous_expires_at expires_at  days_since_expiry  \
0       2016-10-23                 NaT 2018-02-06                NaN   
1395960 2017-04-01          2018-02-06 2017-04-01             -311.0   
1       2017-03-15                 NaT 2017-04-15                NaN   
1395961 2017-04-01          2017-04-15 2017-04-01              -14.0   
2       2017-02-28                 NaT 2017-04-19                NaN   
3       2017-03-31          2017-04-19 2017-05-19              -19.0   
1395962 2017-04-01          2017-05-19 2017-04-01              -48.0   
4       2017-03-26                 NaT 2017-04-26                NaN   
1395963 2017-04-01          2017-04-26 2017-04-01              -25.0   
5       2017-03-15                 NaT 2017-04-15                NaN   

         churn_event  
0                  0  
1395960            0  
1                  0  
1395961            0  
2                  0  
3                  0  
1395962            0 

### Move Churn Flag to Previous Transaction

Churn flag is on the reactivation transaction, but churn logically happened at the previous transaction (that expired and wasn't renewed).

```
Before:
  Trans 3: expires 2015-04-15, churn_event=0
  Trans 4: expires 2015-12-24, churn_event=1 ← Flag is here

After:
  Trans 3: expires 2015-04-15, churned=1 ← Flag moved here ✓
  Trans 4: expires 2015-12-24, churned=0
```

In [82]:
# Move churn flag backward
meaningful_only['churned'] = meaningful_only.groupby('msno')['churn_event'].shift(-1)

print("\nWith adjusted churn flags:")
print(meaningful_only[[
    'trans_at', 'expires_at', 'days_since_expiry', 'churn_event', 'churned'
]].head(20))


With adjusted churn flags:
          trans_at expires_at  days_since_expiry  churn_event  churned
0       2016-10-23 2018-02-06                NaN            0      0.0
1395960 2017-04-01 2017-04-01             -311.0            0      NaN
1       2017-03-15 2017-04-15                NaN            0      0.0
1395961 2017-04-01 2017-04-01              -14.0            0      NaN
2       2017-02-28 2017-04-19                NaN            0      0.0
3       2017-03-31 2017-05-19              -19.0            0      0.0
1395962 2017-04-01 2017-04-01              -48.0            0      NaN
4       2017-03-26 2017-04-26                NaN            0      0.0
1395963 2017-04-01 2017-04-01              -25.0            0      NaN
5       2017-03-15 2017-04-15                NaN            0      0.0
1395964 2017-04-01 2017-04-01              -14.0            0      NaN
6       2017-02-28 2017-04-23                NaN            0      0.0
7       2017-03-31 2017-05-23              -23.0 

### Remove Dummy Transactions

The dummy transaction served its purpose (implementing grace period). Now we remove it by filtering out rows where `churned` is NaN (these are dummy transactions)

In [83]:
# Remove dummy transactions (churned is NaN)
membership_history = meaningful_only[meaningful_only['churned'].notna()].copy()

# Select final columns
membership_history = membership_history[['msno', 'trans_at', 'expires_at', 'churned']]

print(f"\n Final membership history: {len(membership_history):,} records")
print(f"  Covering {membership_history['msno'].nunique():,} users")

# Summary statistics
churn_count = (membership_history['churned'] == 1).sum()
print(f"\nChurn events identified: {churn_count:,}")
print(f"Churn rate per transaction: {churn_count/len(membership_history)*100:.1f}%")

print("\nSample of final history:")
print(membership_history.head(20))


 Final membership history: 1,364,723 records
  Covering 1,171,383 users

Churn events identified: 499
Churn rate per transaction: 0.0%

Sample of final history:
                                            msno   trans_at expires_at  \
0   +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2016-10-23 2018-02-06   
1   +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-03-15 2017-04-15   
2   +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-02-28 2017-04-19   
3   +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-03-31 2017-05-19   
4   +++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= 2017-03-26 2017-04-26   
5   ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= 2017-03-15 2017-04-15   
6   ++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= 2017-02-28 2017-04-23   
7   ++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= 2017-03-31 2017-05-23   
8   ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g= 2017-02-28 2017-04-04   
9   ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g= 2017-03-31 2017-05-04   
10  ++0+IdHga8fCSioOVpU8

## STEP 4 - Derive Subscription Info

We now have a membership history with churn flags on individual transactions. But what we really want for survival analysis is a subscription-level dataset where each row represents a complete subscription period from start to end.

- One row per subscription period
- Clear start and end dates
- Duration calculation
- Multiple subscriptions per user if they churn and return

### The Logic

A subscription is a continuous period from:
- Start: First transaction in the period
- End: When user churns (or NULL if still active) One user can have multiple subscriptions if they churn and come back

**How we find subscriptions:**
1. For each transaction, identify its associated churn date (next churn after this transaction)
2. Group by (user, churn_date) and take the earliest transaction = subscription start
3. Calculate duration and add metadata

### Associate Each Transaction with Its Churn Date

In [85]:
# We start with membership_history from Step 3
print(f"\nMembership history: {len(membership_history):,} events")
print(f"Sample:")
print(membership_history.head(10))


# Create two copies:
all_events = membership_history.copy()
churn_events = membership_history[membership_history['churned'] == 1].copy()

print(f"\nAll events: {len(all_events):,}")
print(f"Churn events only: {len(churn_events):,}") 


Membership history: 1,364,723 events
Sample:
                                           msno   trans_at expires_at  churned
0  +++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s= 2016-10-23 2018-02-06      0.0
1  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= 2017-03-15 2017-04-15      0.0
2  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-02-28 2017-04-19      0.0
3  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw= 2017-03-31 2017-05-19      0.0
4  +++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= 2017-03-26 2017-04-26      0.0
5  ++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= 2017-03-15 2017-04-15      0.0
6  ++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= 2017-02-28 2017-04-23      0.0
7  ++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= 2017-03-31 2017-05-23      0.0
8  ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g= 2017-02-28 2017-04-04      0.0
9  ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g= 2017-03-31 2017-05-04      0.0

All events: 1,364,723
Churn events only: 499


For every transaction, find the next churn event for that user. 

In [86]:
# Prepare churn events (only keep churned transactions)
churn_events_lookup = churn_events[['msno', 'expires_at']].copy()
churn_events_lookup.columns = ['msno', 'churns_at']

print("\nChurn events lookup table:")
print(churn_events_lookup.head())

# For each row in all_events, find the minimum churn date >= trans_at
result_rows = []

for idx, row in all_events.iterrows():
    if idx % 50000 == 0:
        print(f"  Processed {idx:,} / {len(all_events):,} transactions...")
    
    msno = row['msno']
    trans_at = row['trans_at']
    expires_at = row['expires_at']
    churned = row['churned']
    
    # Find churn events for this user that occur on/after this transaction
    user_churns = churn_events_lookup[churn_events_lookup['msno'] == msno]
    future_churns = user_churns[user_churns['churns_at'] >= trans_at]
    
    # Take the earliest (minimum) churn date
    if len(future_churns) > 0:
        churns_at = future_churns['churns_at'].min()
    else:
        churns_at = None  # No churn after this transaction (still active)
    
    result_rows.append({
        'msno': msno,
        'trans_at': trans_at,
        'expires_at': expires_at,
        'churned': churned,
        'churns_at': churns_at
    })

transactions_with_churn_dates = pd.DataFrame(result_rows)

print(f"\nAssociated {len(transactions_with_churn_dates):,} transactions with churn dates")
print("\nSample with churn dates:")
print(transactions_with_churn_dates.head(20))


Churn events lookup table:
                                               msno  churns_at
11205  +Upj5F/6pb1HWrcbR1ayDwdKk3EUliHwEATitoimHQg= 2017-03-01
16599  +kqikTzMDrFeBQX0lsrqr+v8o1yYmwvNVLlYXOBNhXk= 2017-03-01
17158  +mPM354toRZpHddLBeyM7PWmWMjDoR6PmEvQHYhfW4A= 2017-03-01
19435  +tO6SsBJTMwKpdzoCu3/lztEsR7YfJ2pIsDPRBgV5CA= 2017-03-01
19859  +umumRhOZ0IVup9DS+caJIgZNks+ZiGJbss3GAhB8TM= 2017-03-01
  Processed 0 / 1,364,723 transactions...
  Processed 50,000 / 1,364,723 transactions...
  Processed 100,000 / 1,364,723 transactions...
  Processed 150,000 / 1,364,723 transactions...
  Processed 200,000 / 1,364,723 transactions...
  Processed 250,000 / 1,364,723 transactions...
  Processed 300,000 / 1,364,723 transactions...
  Processed 350,000 / 1,364,723 transactions...
  Processed 400,000 / 1,364,723 transactions...
  Processed 450,000 / 1,364,723 transactions...
  Processed 500,000 / 1,364,723 transactions...
  Processed 550,000 / 1,364,723 transactions...
  Processed 600,000 / 1,3

### Group Transactions into Subscriptions

Find the first transaction for each subscription period.

In [87]:
# Group by (msno, churns_at) and take MIN(trans_at) as subscription start
subscriptions = transactions_with_churn_dates.groupby(['msno', 'churns_at']).agg({
    'trans_at': 'min'  # Earliest transaction = subscription start
}).reset_index()

subscriptions.columns = ['msno', 'ends_at', 'starts_at']

# Reorder columns
subscriptions = subscriptions[['msno', 'starts_at', 'ends_at']]

print(f"Created {len(subscriptions):,} subscriptions")
print(f"From {subscriptions['msno'].nunique():,} unique users")

Created 499 subscriptions
From 499 unique users


### Calculate Duration and Metadata

In [88]:
# Sort and add subscription_id
subscriptions = subscriptions.sort_values(['msno', 'starts_at']).reset_index(drop=True)
subscriptions['subscription_id'] = range(1, len(subscriptions) + 1)

# Calculate duration_days
observation_end = pd.Timestamp('2017-04-01')

subscriptions['duration_days'] = subscriptions.apply(
    lambda row: (
        (row['ends_at'] if pd.notna(row['ends_at']) else observation_end) - row['starts_at']
    ).days + 1,  # +1 to include both start and end day
    axis=1
)

# Churned flag: 1 if ends_at is not NULL, 0 if NULL (censored)
subscriptions['churned'] = subscriptions['ends_at'].notna().astype(int)

# Count prior subscriptions for each user
subscriptions['prior_subscriptions'] = subscriptions.groupby('msno').cumcount()

print(f"\nTotal subscriptions: {len(subscriptions):,}")
print(f"Unique users: {subscriptions['msno'].nunique():,}")

print(f"\nChurn distribution:")
churn_counts = subscriptions['churned'].value_counts()
print(f"  Churned (1): {churn_counts.get(1, 0):,} ({churn_counts.get(1, 0)/len(subscriptions)*100:.1f}%)")
print(f"  Active/Censored (0): {churn_counts.get(0, 0):,} ({churn_counts.get(0, 0)/len(subscriptions)*100:.1f}%)")

print(f"\nDuration statistics (days):")
print(subscriptions['duration_days'].describe())

print(f"\nPrior subscriptions distribution:")
print(subscriptions['prior_subscriptions'].value_counts().sort_index().head(10))

# Users with multiple subscriptions
multi_sub_users = subscriptions.groupby('msno').size()
multi_sub_users = multi_sub_users[multi_sub_users > 1]
print(f"\nUsers with multiple subscriptions: {len(multi_sub_users):,}")
if len(multi_sub_users) > 0:
    print(f"  Max subscriptions per user: {multi_sub_users.max()}")

print("\nSample of final subscriptions dataset:")
print(subscriptions[['subscription_id', 'msno', 'starts_at', 'ends_at', 
                      'duration_days', 'churned', 'prior_subscriptions']].head(20))


Total subscriptions: 499
Unique users: 499

Churn distribution:
  Churned (1): 499 (100.0%)
  Active/Censored (0): 0 (0.0%)

Duration statistics (days):
count    499.000000
mean       4.160321
std       40.080170
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max      568.000000
Name: duration_days, dtype: float64

Prior subscriptions distribution:
prior_subscriptions
0    499
Name: count, dtype: int64

Users with multiple subscriptions: 0

Sample of final subscriptions dataset:
    subscription_id                                          msno  starts_at  \
0                 1  +Upj5F/6pb1HWrcbR1ayDwdKk3EUliHwEATitoimHQg= 2017-03-01   
1                 2  +kqikTzMDrFeBQX0lsrqr+v8o1yYmwvNVLlYXOBNhXk= 2017-03-01   
2                 3  +mPM354toRZpHddLBeyM7PWmWMjDoR6PmEvQHYhfW4A= 2017-03-01   
3                 4  +tO6SsBJTMwKpdzoCu3/lztEsR7YfJ2pIsDPRBgV5CA= 2017-03-01   
4                 5  +umumRhOZ0IVup9DS+caJIgZNks+ZiGJbss3GAhB8TM= 2017-03-01   
5 

## Export Results

Save the final subscriptions dataset for survival analysis.

In [89]:
# Save to CSV
subscriptions.to_csv('kkbox_subscriptions.csv', index=False)
print("\nSubscriptions saved to 'kkbox_subscriptions.csv'")

# Save to pickle for faster loading
subscriptions.to_pickle('kkbox_subscriptions.pkl')
print("Subscriptions saved to 'kkbox_subscriptions.pkl'")

# Also save membership history (might be useful)
membership_history.to_csv('kkbox_membership_history.csv', index=False)
print("Membership history saved to 'kkbox_membership_history.csv'")


Subscriptions saved to 'kkbox_subscriptions.csv'
Subscriptions saved to 'kkbox_subscriptions.pkl'
Membership history saved to 'kkbox_membership_history.csv'


We now have a clean subscriptions dataset ready for:
- Kaplan-Meier survival curves
- Cox Proportional Hazards regression
- Churn prediction modeling